In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dotenv import find_dotenv

ROOT = Path(find_dotenv()).parent
RAW_DIR = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

files = sorted(RAW_DIR.glob("pm25_*.csv"))
print("Files found:", len(files))
for f in files:
    print(" ", f.name)

Files found: 4
  pm25_17_12234787.csv
  pm25_235_12235610.csv
  pm25_50_12234796.csv
  pm25_8118_23534.csv


In [4]:
frames, summary = [], []
for f in files:
    df = pd.read_csv(f)
    station = df["station"].iloc[0]
    df["time_utc"] = pd.to_datetime(df["time_utc"], utc=True)

    # 1. remove impossible values
    df = df[(df["pm25"] >= 0) & (df["pm25"] <= 1000)]

    # 2. convert to Indian time and sort
    df["time"] = df["time_utc"].dt.tz_convert("Asia/Kolkata")
    s = df.set_index("time")["pm25"].sort_index()
    s = s[~s.index.duplicated()]

    # 3. make sure every hour has a row (gaps become NaN = "not a number")
    full = s.resample("h").mean()
    missing_before = full.isna().mean() * 100

    # 4. fill only short gaps (up to 3 hours) by drawing a line between neighbours
    full = full.interpolate(limit=3)
    missing_after = full.isna().mean() * 100

    out = full.rename("pm25").reset_index()
    out["station"] = station
    frames.append(out)
    summary.append({
        "station": station,
        "hours": len(full),
        "from": full.index.min().date(),
        "to": full.index.max().date(),
        "missing_before_%": round(missing_before, 1),
        "missing_after_%": round(missing_after, 1),
    })

all_clean = pd.concat(frames, ignore_index=True)
all_clean.to_csv(PROC_DIR / "pm25_hourly_clean.csv", index=False)
pd.DataFrame(summary)


,station,hours,from,to,missing_before_%,missing_after_%
0,"R K Puram, Delhi - DPCC",8734,2025-10-15,2026-10-13,12.4,7.4
1,"Anand Vihar, New Delhi - DPCC",8734,2025-10-15,2026-10-13,13.4,8.4
2,"Punjabi Bagh, Delhi - DPCC",8648,2025-10-18,2026-10-13,12.0,6.8
3,New Delhi,7386,2025-10-15,2026-08-18,5.2,3.7
